# 예제 04. 모델 클래스 작성
빅데이터프로그래밍 · 6주차

## 목표
- `nn.Module` 을 상속해 모델을 만든다
- `__init__` 에 층을 만들고 `forward()` 에 흐름을 쓴다
- Sequential과 클래스 방식의 차이를 이해한다

5주차 Dataset의 세 메서드와 같은 구조입니다 — 정해진 자리에 정해진 내용을 씁니다.


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)


## 1. 가장 기본 형태

| 위치 | 하는 일 |
| --- | --- |
| `__init__` | 층을 만들어 둔다 |
| `forward` | 데이터가 흐르는 순서를 쓴다 |

`forward()` 는 직접 부르지 않습니다. `model(x)` 라고 쓰면 PyTorch가 대신 부릅니다.


In [ ]:
class SimpleNet(nn.Module):
    def __init__(self, in_features=4, hidden=16, out_features=1):
        super().__init__()                       # 반드시 먼저
        self.fc1 = nn.Linear(in_features, hidden)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden, out_features)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


model = SimpleNet()
print(model)


In [ ]:
x = torch.randn(5, 4)
print("입력:", x.shape)
print("출력:", model(x).shape)
print("파라미터:", sum(p.numel() for p in model.parameters()))


## 2. forward 안에 print를 넣어 흐름 보기


In [ ]:
class LoudNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.fc2 = nn.Linear(8, 2)

    def forward(self, x):
        print("  입력      ", tuple(x.shape))
        x = torch.relu(self.fc1(x))
        print("  fc1+relu  ", tuple(x.shape))
        x = self.fc2(x)
        print("  fc2       ", tuple(x.shape))
        return x


print("model(x) 호출:")
_ = LoudNet()(torch.randn(3, 4))


## 3. 은닉층 1개 vs 2개 — 과제와 같은 비교


In [ ]:
class OneHidden(nn.Module):
    def __init__(self, in_f=4, h=16, out_f=3):
        super().__init__()
        self.fc1 = nn.Linear(in_f, h)
        self.fc2 = nn.Linear(h, out_f)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))


class TwoHidden(nn.Module):
    def __init__(self, in_f=4, h1=16, h2=16, out_f=3):
        super().__init__()
        self.fc1 = nn.Linear(in_f, h1)
        self.fc2 = nn.Linear(h1, h2)
        self.fc3 = nn.Linear(h2, out_f)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)


import pandas as pd

rows = []
for name, m in [("은닉층 1개", OneHidden()), ("은닉층 2개", TwoHidden())]:
    rows.append({
        "모델": name,
        "Linear 층 수": sum(1 for l in m.modules() if isinstance(l, nn.Linear)),
        "파라미터 수": sum(p.numel() for p in m.parameters()),
        "출력 shape": tuple(m(torch.randn(5, 4)).shape),
    })
pd.DataFrame(rows)


## 4. 분류 모델로 만들기
출력 뉴런 수 = 분류할 클래스 수. 손실 함수는 `CrossEntropyLoss` 를 씁니다.


In [ ]:
class Classifier(nn.Module):
    def __init__(self, in_f=4, h=32, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_f, h), nn.ReLU(),
            nn.Linear(h, h), nn.ReLU(),
            nn.Linear(h, n_classes),
        )

    def forward(self, x):
        return self.net(x)


clf = Classifier()
print(clf)

x = torch.randn(6, 4)
logits = clf(x)
print("\nlogits:", logits.shape)
print("확률   :", torch.softmax(logits, dim=1)[0].data.round(decimals=3))
print("예측   :", logits.argmax(dim=1).tolist())


`CrossEntropyLoss` 는 softmax를 안에서 계산합니다. 모델 마지막에 softmax를 넣지 않습니다.


In [ ]:
loss_fn = nn.CrossEntropyLoss()
y = torch.randint(0, 3, (6,))              # 정답은 클래스 번호
print("정답:", y.tolist())
print("손실:", loss_fn(logits, y).item())


## 5. 모델 구조 확인하기 — 세 가지 방법


In [ ]:
print("① print(model)")
print(clf)

print("\n② 층 목록")
for name, module in clf.named_children():
    print(" ", name, module.__class__.__name__)

print("\n③ 파라미터별 크기")
for name, p in clf.named_parameters():
    print(f"  {name:14s} {tuple(p.shape)}")


## 6. 장치로 옮기기 — 4주차 규칙
모델과 데이터가 같은 장치에 있어야 합니다.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
clf = clf.to(device)

x = torch.randn(6, 4).to(device)
print("device:", device, "/ 출력:", clf(x).shape)


## 직접 해보기
1. 은닉층이 3개인 모델을 클래스로 작성하고 파라미터 수를 세세요.
2. `forward()` 에서 ReLU를 빼면 출력이 어떻게 달라지나요?
3. `super().__init__()` 을 빼고 실행하면 어떤 오류가 나나요?


In [ ]:
# 여기에 작성하세요
